# FastSpeech 2 — English (Svarah) & Telugu (IndicTTS) on Google Colab

1. Runtime → Change runtime type → **GPU** (T4 is fine).
2. Upload `FastSpeech2_project.zip` to the root of your Google Drive.
3. Set `LANG` below to `English` or `Telugu` and run all cells. Run the notebook twice (or in two Colab tabs) for the two languages.

Everything lives on Google Drive, so if Colab disconnects just re-run all cells — training resumes from `checkpoints/latest.pth`.

In [ ]:
LANG = 'English'   # or 'Telugu'
MAX_STEPS = 50000   # lower (e.g. 30000) if short on time
HF_TOKEN = ''       # needed for English (Svarah is gated): huggingface.co/settings/tokens, after accepting terms at huggingface.co/datasets/ai4bharat/Svarah

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
ROOT = '/content/drive/MyDrive/FastSpeech2_project'
if not os.path.exists(ROOT):
    !unzip -q /content/drive/MyDrive/FastSpeech2_project.zip -d /content/drive/MyDrive/
%cd {ROOT}
!pip -q install -r requirements.txt

Raw/preprocessed audio is kept on the local Colab disk (fast), checkpoints and results on Drive. If the runtime resets, cells re-download and re-preprocess the data (≈15–30 min) and training resumes from the last checkpoint.

In [ ]:
import os
os.environ['NLTK_ALLOW_PROXIED_URLOPEN']='1'
os.makedirs('/content/data', exist_ok=True)
if not os.path.islink(f'{ROOT}/data'):
    if os.path.exists(f'{ROOT}/data'): os.rename(f'{ROOT}/data', f'{ROOT}/data_old')
    os.symlink('/content/data', f'{ROOT}/data')
%cd {ROOT}/{LANG}
tok = f'--hf_token {HF_TOKEN}' if HF_TOKEN else ''
if not os.path.exists(f'../data/{LANG}/raw/train.tsv'):
    !python prepare_data.py {tok}
!cat ../data/{LANG}/raw/selection_stats.json

In [ ]:
if not os.path.exists(f'../data/{LANG}/preprocessed/train.txt'):
    !python preprocess.py --workers 2
!cat ../data/{LANG}/preprocessed/summary.json

In [ ]:
!python train.py --max_steps {MAX_STEPS} --workers 2

In [ ]:
!python inference.py
from IPython.display import Audio, Image, display
import glob
for w in sorted(glob.glob(f'../Results/{LANG}/test_generated_audios/*.wav')):
    print(w); display(Audio(w))

In [ ]:
display(Image(f'../Results/{LANG}/training_loss.png'))
display(Image(f'../Results/{LANG}/validation_loss.png'))